# HAR-RV and Ensemble Models for Volatility forecasting

### HAR-RV = Heterogeneous AutoRegressive model for Realized Volatility

Financial markets have participants operating on very different time horizons simultaneously — a high-frequency market maker cares about volatility over the next hour, a hedge fund over the next week, a pension fund over the next month. Each group's trading behavior generates volatility at their own scale, and these scales interact.
Standard GARCH models have one "memory" — the single α+β persistence parameter. They can't separately capture short-term clustering and long-term mean reversion at the same time. A GARCH model fit to daily data essentially averages these different components into one number and loses information.

It models future realized variance/volatility measured over multiple time scales:

$$ RV_{t+1} = \beta_0 + \beta_d RV^d_{t} + \beta_w \overline{RV}^w_{t-4:t} + \beta_m \overline{RV}^m_{t-21:t} + \epsilon_{t+1} $$

where
* $ RV^d_{t-1}$ is yesterday's realized variance (1-day)

* $ \overline{RV}^w_{t-1}$ is average of the pass week's(5 days) realized variance

* $ \overline{RV}^m_{t-1}$ is the average of the past month's (22 days) realized variance

It is essentially an OLS regression with three features. It works because it matches the empirical structure of volatility.

**Volatility has long memory:** If you plot the autocorrelation of daily realized variance, it decays extremely slowly-- there is still meaningful autocorrelation at lags of 50, 100, even 200 days.

GARCH(1,1) has short memory by construction; it's autocorrelation decays geometrically fast. HAR approximated long memory by stacking components at three horizons, which empirically captures the slow decay well without requiring a complex model.

HAR is linear. This means it is fast, interpretable, and you can derive prediction intervals. The coefficients are economically meaningful -- typically $ \beta_d > \beta_w > \beta_m > 0$, meaning recent volatility matters most but all three horizons contribute. It usually beats GARCH out-of-sample. The reason is that the 5-day and 22-day components act as a kind of regularization — they smooth out noise in the 1-day component and give the model a better estimate of the "volatility regime" you're currently in.

In [34]:
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
# ! pip install pyarrow
import pyarrow

sys.path.append(r"C:\Users\arbaz2\Desktop\Quant Finance\Volatility Forecasting\src")

from data_pipeline import make_dataset
from baselines import make_baseline_forecasts
from metrics import qlike, mse, score, score_by_regime, score_by_ticker
from garch import add_garch_refit_recurse, add_gjr_garch_forecast, plot_vol_compare

CRISIS_WINDOWS = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31"),
}



In [4]:
df = make_dataset(["SPY", "JPM"], start="2000-01-01", horizons=(1,5), crisis_windows=CRISIS_WINDOWS)
df["date"] = pd.to_datetime(df["date"])

## Plotting the AutoCorrelation of daily realized volatility to show that it has long memory

* AutoCorrelation Function (ACF) at lag 1 should be strong
* ACF at lag 5-7 (a week) should still be meaningful
* ACF at lag 21 (a month) and even 100 is often nontrivial

In [5]:
import plotly.graph_objects as go

def plot_rv_acf(df, ticker, max_lag=150):
    """
    Plot autocorrelation of daily realized variance rv1_var
    for one ticker.
    """
    d = df[df["ticker"] == ticker].sort_values("date").copy()
    x = d["rv1_var"].dropna()

    acf_vals = [x.autocorr(lag=lag) for lag in range(1, max_lag + 1)]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=list(range(1, max_lag + 1)),
        y=acf_vals,
        name="ACF"
    ))

    # highlight important lags
    for lag in [1, 7, 21, 100]:
        if lag <= max_lag:
            fig.add_vline(x=lag, line_dash="dash", line_color="red")

    fig.update_layout(
        title=f"{ticker}: Autocorrelation of Daily Realized Variance",
        xaxis_title="Lag (days)",
        yaxis_title="Autocorrelation"
    )

    fig.show()

In [6]:
plot_rv_acf(df, "SPY", max_lag=150)
plot_rv_acf(df, "JPM", max_lag=150)

In [10]:
df_f = make_baseline_forecasts(df, hv_window=20, ewma_lam=0.94)
df_f.head(10)

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389448,4723500,-1.733113,3.003682,5.704863,14.393872,calm,NaN,NaN,5.291806,26.459031
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641846,5741700,0.342420,0.117251,1.448977,36.092576,calm,NaN,NaN,1.468517,7.342585
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861015,8405550,-2.388486,5.704863,0.390316,14.400736,calm,NaN,NaN,5.154519,25.772593
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545334,7503700,-1.203735,1.448977,0.999674,37.060314,calm,NaN,NaN,1.387441,6.937206
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998020,7271850,0.624753,0.390316,2.253448,14.666889,calm,NaN,NaN,5.187539,25.937697
5,2000-01-12,SPY,144.593750,144.593750,142.875000,143.062500,89.644539,6907700,-0.999837,0.999674,1.809667,36.244830,calm,NaN,NaN,1.391133,6.955666
6,2000-01-13,JPM,47.416668,48.333332,47.041668,47.541668,22.330734,6918900,1.501149,2.253448,12.463047,23.815356,calm,NaN,NaN,4.899706,24.498530
7,2000-01-13,SPY,144.468750,145.750000,143.281250,145.000000,90.858620,5158300,1.345239,1.809667,1.818881,6.194450,calm,NaN,NaN,1.367646,6.838229
8,2000-01-14,JPM,49.291668,50.500000,48.541668,49.250000,23.133158,9731850,3.530304,12.463047,15.756988,36.568663,calm,NaN,NaN,4.740931,23.704653
9,2000-01-14,SPY,146.531250,147.468750,145.968750,146.968750,92.092293,7437300,1.348659,1.818881,0.623902,6.701100,calm,NaN,NaN,1.394167,6.970835


In [11]:
df_garch = add_garch_refit_recurse(df_f, refit_every=5, min_train=750, mean="zero", dist="t")
df_garch.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var,garch1_var,garch5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389448,4723500,-1.733113,3.003682,5.704863,14.393872,calm,NaN,NaN,5.291806,26.459031,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641846,5741700,0.342420,0.117251,1.448977,36.092576,calm,NaN,NaN,1.468517,7.342585,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861015,8405550,-2.388486,5.704863,0.390316,14.400736,calm,NaN,NaN,5.154519,25.772593,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545334,7503700,-1.203735,1.448977,0.999674,37.060314,calm,NaN,NaN,1.387441,6.937206,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998020,7271850,0.624753,0.390316,2.253448,14.666889,calm,NaN,NaN,5.187539,25.937697,NaN,NaN


In [12]:
df_gjr_garch = add_gjr_garch_forecast(df_garch, refit_every=21, min_train=750, dist="t" )
df_gjr_garch.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var,garch1_var,garch5_var,gjr1_var,gjr5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389448,4723500,-1.733113,3.003682,...,14.393872,calm,NaN,NaN,5.291806,26.459031,NaN,NaN,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641846,5741700,0.342420,0.117251,...,36.092576,calm,NaN,NaN,1.468517,7.342585,NaN,NaN,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861015,8405550,-2.388486,5.704863,...,14.400736,calm,NaN,NaN,5.154519,25.772593,NaN,NaN,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545334,7503700,-1.203735,1.448977,...,37.060314,calm,NaN,NaN,1.387441,6.937206,NaN,NaN,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998020,7271850,0.624753,0.390316,...,14.666889,calm,NaN,NaN,5.187539,25.937697,NaN,NaN,NaN,NaN


In [13]:
models_1d = ["hv1_var","ewma1_var","garch1_var","gjr1_var"]
print(score(df_gjr_garch, models_1d, "rv1_var", eval_start="2005-01-01"))
print("\n")

models_5d = ["hv5_var", "ewma5_var", "garch5_var","gjr5_var"]
print(score(df_gjr_garch, models_5d, "rv5_var", eval_start="2005-01-01"))
print("\n")
print(score_by_regime(df_gjr_garch, models_1d, "rv1_var", eval_start="2005-01-01").head(30))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10888  1.355589  208.550982
2  garch1_var  rv1_var  10888  1.389883  215.365621
1   ewma1_var  rv1_var  10888  1.445831  221.383209
0     hv1_var  rv1_var  10888  1.486515  230.501813


        model   target      n     QLIKE          MSE
3    gjr5_var  rv5_var  10888  2.829688   978.630534
2  garch5_var  rv5_var  10888  2.842299  1035.557144
0     hv5_var  rv5_var  10888  2.888738  1424.962644
1   ewma5_var  rv5_var  10888  2.909016  1302.078946


           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724734  1546.461162
2      COVID_2020  garch1_var  rv1_var   144  3.968354  1576.126199
0      COVID_2020     hv1_var  rv1_var   144  4.259609  1823.977356
1      COVID_2020   ewma1_var  rv1_var   144  4.591998  1750.481312
7   GFC_2007_2009    gjr1_var  rv1_var  1008  2.948959  1834.599801
6   GFC_2007_2009  garch1_var  rv1_var  1008  3.004557  1900.09

In [17]:
plot_vol_compare(df_gjr_garch, "SPY", target_var="rv1_var", forecast_vars=("hv1_var","ewma1_var","garch1_var", "gjr1_var"), crisis_windows = CRISIS_WINDOWS )

In [18]:
def make_har_features_single(d):
    """
    Create HAR-RV features for a single ticker.

    Uses lagged realized variance at 3 time scales:
    - rv_d : yesterday's realized variance
    - rv_w : average realized variance over last 5 days
    - rv_m : average realized variance over last 22 days
    """
    d = d.sort_values("date").copy()

    # 1-day lag
    d["rv_d"] = d["rv1_var"].shift(1)

    # 5-day average lag
    d["rv_w"] = d["rv1_var"].shift(1).rolling(window=5).mean()

    # 22-day average lag
    d["rv_m"] = d["rv1_var"].shift(1).rolling(window=22).mean()

    return d

In [19]:
def add_har_features(df):
    """
    Apply HAR feature construction separately to each ticker.
    """
    out = df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)

    pieces = []
    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()
        d = make_har_features_single(d)
        pieces.append(d)

    out = pd.concat(pieces, axis=0).sort_values(["date", "ticker"]).reset_index(drop=True)
    return out

In [20]:
''' Reuseable rolling HAR forecaster for any target'''

def har_rolling_forecast_single(
    d,
    target_col="rv1_var",
    refit_every=21,
    min_train=252
):
    """
    Expanding-window HAR forecast for one ticker.

    Parameters
    ----------
    d : DataFrame for one ticker, must contain:
        ['date', 'rv_d', 'rv_w', 'rv_m', target_col]
    target_col : str
        Either 'rv1_var' or 'rv5_var'
    refit_every : int
        Refit frequency in trading days
    min_train : int
        Minimum number of valid training rows
    """
    d = d.sort_values("date").copy()

    feature_cols = ["rv_d", "rv_w", "rv_m"]
    fcast = np.full(len(d), np.nan)

    t = min_train

    while t < len(d):
        # training data up to t
        train = d.iloc[:t].dropna(subset=feature_cols + [target_col]).copy()

        if len(train) < min_train:
            t += refit_every
            continue

        X_train = train[feature_cols].values
        y_train = train[target_col].values

        model = LinearRegression()
        model.fit(X_train, y_train)

        # forecast until next refit
        t_end = min(t + refit_every, len(d))

        for i in range(t, t_end):
            row = d.iloc[i]

            if row[feature_cols].isna().any():
                continue

            X_test = row[feature_cols].values.reshape(1, -1)
            fcast[i] = model.predict(X_test)[0]

        t = t_end

    return pd.Series(fcast, index=d.index)

In [21]:
''' Add HAR forecasts for both 1-day and 5-day horizons'''

def add_har_forecasts(
    df,
    refit_every=21,
    min_train=252
):
    """
    Add proper HAR forecasts for both 1-day and 5-day horizons.

    Adds:
    - har1_var : HAR fitted directly to rv1_var
    - har5_var : HAR fitted directly to rv5_var
    """
    out = add_har_features(df)
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    har1_list = []
    har5_list = []

    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()

        s1 = har_rolling_forecast_single(
            d,
            target_col="rv1_var",
            refit_every=refit_every,
            min_train=min_train
        )

        s5 = har_rolling_forecast_single(
            d,
            target_col="rv5_var",
            refit_every=refit_every,
            min_train=min_train
        )

        har1_list.append(s1.rename(tkr))
        har5_list.append(s5.rename(tkr))

    har1_all = pd.concat(har1_list, axis=0).sort_index()
    har5_all = pd.concat(har5_list, axis=0).sort_index()

    out["har1_var"] = har1_all
    out["har5_var"] = har5_all

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [22]:
df_har = add_har_forecasts(df_gjr_garch, refit_every=21, min_train=252)

In [23]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var", "gjr1_var", "har1_var"]
print(score(df_har, models_1d, "rv1_var", eval_start="2005-01-01"))
print()

models_5d = ["hv5_var", "ewma5_var", "garch5_var", "gjr5_var", "har5_var"]
print(score(df_har, models_5d, "rv5_var", eval_start="2005-01-01"))
print()

print(score_by_regime(df_har, models_1d, "rv1_var", eval_start="2005-01-01"))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10888  1.355589  208.550982
2  garch1_var  rv1_var  10888  1.389883  215.365621
4    har1_var  rv1_var  10888  1.442817  218.390358
1   ewma1_var  rv1_var  10888  1.445831  221.383209
0     hv1_var  rv1_var  10888  1.486515  230.501813

        model   target      n     QLIKE          MSE
4    har5_var  rv5_var  10888  2.728862   369.222427
3    gjr5_var  rv5_var  10888  2.829688   978.630534
2  garch5_var  rv5_var  10888  2.842299  1035.557144
0     hv5_var  rv5_var  10888  2.888738  1424.962644
1   ewma5_var  rv5_var  10888  2.909016  1302.078946

           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724734  1546.461162
4      COVID_2020    har1_var  rv1_var   144  3.942688  1462.861860
2      COVID_2020  garch1_var  rv1_var   144  3.968354  1576.126199
0      COVID_2020     hv1_var  rv1_var   144  4.259609  1823.977356
1      COVID_2020   ewma1_va

In [24]:
plot_vol_compare(df_har, "SPY", target_var="rv1_var", forecast_vars=("hv1_var","ewma1_var","garch1_var", "gjr1_var","har1_var"), crisis_windows = CRISIS_WINDOWS )

We can see from the scores above that GJR-GARCH wins overall which makes sense considering it is the most sophisticated parametric model and captures the leverage effect. HAR-RV performs roughly similar to EWMA. ARCH-family models clearly dominate simple smoothing models. This is expected when realized variance is noisy, which it is when computed from daily returns.

* For 5-day forecast: HAR is the best model for 5-day horizon as it captures long memory and models volatility persistence across time scales. GARCH models are optimized for short-horizon conditional variance and assume exponential decay of memory and are therefore behind HAR.

* For COVID: GJR wins comfortably, HAR comes second, GARCH third, and EWMA is worst. It makes sense as it was a sudden shock where leverage matters most and therefore the asymmetric model wins.

* For GFC: EWMA and HAR as essentially tied which makes sense as GFC was a prolonged crisis lasting nearly two years, so EWMA's exponential weighting adapts reasonably well to the sustained high-volatility regime. HAR's 22-day component also eventually incorporates the crisis.

# Considering an Ensemble of Models

#### We can also try a simple ensemble as
$$ \sigma^2_{ensemble}  = 0.5 \times GJR + 0.5\times HAR $$

In [25]:
df_all = df_har.copy()
df_all["ens1_var"] = 0.5 * df_har["gjr1_var"] + 0.5 * df_har["har1_var"]
df_all["ens5_var"] = 0.5 * df_har["gjr5_var"] + 0.5 * df_har["har5_var"]
df_all.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,garch5_var,gjr1_var,gjr5_var,rv_d,rv_w,rv_m,har1_var,har5_var,ens1_var,ens5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389448,4723500,-1.733113,3.003682,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641846,5741700,0.342420,0.117251,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861015,8405550,-2.388486,5.704863,...,NaN,NaN,NaN,5.704863,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545334,7503700,-1.203735,1.448977,...,NaN,NaN,NaN,1.448977,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998020,7271850,0.624753,0.390316,...,NaN,NaN,NaN,0.390316,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var", "gjr1_var", "har1_var", "ens1_var"]
print(score(df_all, models_1d, "rv1_var", eval_start="2005-01-01"))
print()

models_5d = ["hv5_var", "ewma5_var", "garch5_var", "gjr5_var", "har5_var", "ens5_var"]
print(score(df_all, models_5d, "rv5_var", eval_start="2005-01-01"))
print()

print(score_by_regime(df_all, models_1d, "rv1_var", eval_start="2005-01-01"))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10888  1.355589  208.550982
5    ens1_var  rv1_var  10888  1.375676  210.118022
2  garch1_var  rv1_var  10888  1.389883  215.365621
4    har1_var  rv1_var  10888  1.442817  218.390358
1   ewma1_var  rv1_var  10888  1.445831  221.383209
0     hv1_var  rv1_var  10888  1.486515  230.501813

        model   target      n     QLIKE          MSE
4    har5_var  rv5_var  10888  2.728862   369.222427
5    ens5_var  rv5_var  10888  2.760539   497.785326
3    gjr5_var  rv5_var  10888  2.829688   978.630534
2  garch5_var  rv5_var  10888  2.842299  1035.557144
0     hv5_var  rv5_var  10888  2.888738  1424.962644
1   ewma5_var  rv5_var  10888  2.909016  1302.078946

           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724734  1546.461162
5      COVID_2020    ens1_var  rv1_var   144  3.750110  1478.389793
4      COVID_2020    har1_var  rv1_var   144  3.942688  146

### In general we can have
$$ \hat{\sigma}^2_{ensemble}  = w \hat{\sigma}^2_{GJR} + (1-w) \hat{\sigma}^2_{HAR} $$

where the weight $w$ can be determined through optimization e.g., minimizing QLIKE over $w$.

To do this we use three periods:
* Train/model estimation: Already handled inside GJR and HAR fits
* Development window to optimize $w$ (e.g., 2005-01-01 to 2006-12-31)
* Evaluation window to report final performance (e.g., 2007-01-01 onwards)

In [27]:
def qlike_loss(y_true, y_pred, eps=1e-12):
    """
    Mean QLIKE loss for variance forecasts.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_true = np.maximum(y_true, eps)
    y_pred = np.maximum(y_pred, eps)

    return np.mean(np.log(y_pred) + y_true / y_pred)


def optimize_ensemble_weight(
    df,
    model_a_col,
    model_b_col,
    target_col,
    start="2005-01-01",
    end="2006-12-31",
    weight_grid=None
):
    """
    Optimize ensemble weight w in:

        ensemble = w * model_a + (1 - w) * model_b

    by minimizing QLIKE on a development window.
    """
    if weight_grid is None:
        weight_grid = np.linspace(0.0, 1.0, 101)

    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])

    # keep only development window
    d = d[
        (d["date"] >= pd.to_datetime(start)) &
        (d["date"] <= pd.to_datetime(end))
    ].copy()

    # keep only rows where both models and target exist
    d = d.dropna(subset=[model_a_col, model_b_col, target_col]).copy()

    best_w = None
    best_loss = np.inf
    rows = []

    for w in weight_grid:
        ens = w * d[model_a_col].values + (1.0 - w) * d[model_b_col].values
        loss = qlike_loss(d[target_col].values, ens)

        rows.append({"w": w, "qlike": loss})

        if loss < best_loss:
            best_loss = loss
            best_w = w

    res_df = pd.DataFrame(rows)
    return best_w, best_loss, res_df



def add_weighted_ensemble(df, w1, w5):
    """
    Add optimized ensemble forecasts:
      ens1_var = w1 * gjr1_var + (1-w1) * har1_var
      ens5_var = w5 * gjr5_var + (1-w5) * har5_var
    """
    out = df.copy()

    out["ens1_var"] = w1 * out["gjr1_var"] + (1.0 - w1) * out["har1_var"]
    out["ens5_var"] = w5 * out["gjr5_var"] + (1.0 - w5) * out["har5_var"]

    return out

In [28]:
# 1-day ensemble weight
w1_star, loss1_star, w1_table = optimize_ensemble_weight(
    df_har,
    model_a_col="gjr1_var",
    model_b_col="har1_var",
    target_col="rv1_var",
    start="2005-01-01",
    end="2006-12-31"
)

print("Optimal w1 (GJR weight for 1-day):", w1_star)
print("Best 1-day dev QLIKE:", loss1_star)


# 5-day ensemble weight
w5_star, loss5_star, w5_table = optimize_ensemble_weight(
    df_har,
    model_a_col="gjr5_var",
    model_b_col="har5_var",
    target_col="rv5_var",
    start="2005-01-01",
    end="2006-12-31"
)

print("Optimal w5 (GJR weight for 5-day):", w5_star)
print("Best 5-day dev QLIKE:", loss5_star)

Optimal w1 (GJR weight for 1-day): 1.0
Best 1-day dev QLIKE: 0.4638888119198429
Optimal w5 (GJR weight for 5-day): 0.0
Best 5-day dev QLIKE: 1.9572051217837039


In [29]:
import plotly.express as px
def plot_weight_search_both(w1_table, w5_table):
    """
    Compare 1-day and 5-day ensemble weight searches on the same figure.
    """
    d1 = w1_table.copy()
    d1["horizon"] = "1-day"

    d5 = w5_table.copy()
    d5["horizon"] = "5-day"

    d = pd.concat([d1, d5], axis=0)

    fig = px.line(
        d, x="w", y="qlike", color="horizon", markers=True,
        title="Ensemble Weight Search: 1-Day vs 5-Day"
    )
    fig.update_layout(
        xaxis_title="Weight on GJR",
        yaxis_title="Development-window QLIKE"
    )
    fig.show()

plot_weight_search_both(w1_table, w5_table)

In [30]:
df_ensemble = add_weighted_ensemble(df_har, w1=w1_star, w5=w5_star)

In [31]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var", "gjr1_var", "har1_var", "ens1_var"]
print(score(df_ensemble, models_1d, "rv1_var", eval_start="2007-01-01"))
print()

models_5d = ["hv5_var", "ewma5_var", "garch5_var", "gjr5_var", "har5_var", "ens5_var"]
print(score(df_ensemble, models_5d, "rv5_var", eval_start="2007-01-01"))
print()

print(score_by_regime(df_ensemble, models_1d, "rv1_var", eval_start="2007-01-01"))

        model   target     n     QLIKE         MSE
3    gjr1_var  rv1_var  9882  1.446365  229.566867
5    ens1_var  rv1_var  9882  1.446365  229.566867
2  garch1_var  rv1_var  9882  1.480823  237.068693
4    har1_var  rv1_var  9882  1.515458  240.259753
1   ewma1_var  rv1_var  9882  1.541592  243.699898
0     hv1_var  rv1_var  9882  1.583052  253.745020

        model   target     n     QLIKE          MSE
5    ens5_var  rv5_var  9882  2.807418   406.243994
4    har5_var  rv5_var  9882  2.807418   406.243994
3    gjr5_var  rv5_var  9882  2.912648  1077.377143
2  garch5_var  rv5_var  9882  2.925092  1140.087752
0     hv5_var  rv5_var  9882  2.976275  1569.110965
1   ewma5_var  rv5_var  9882  2.998532  1433.706430

           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724734  1546.461162
5      COVID_2020    ens1_var  rv1_var   144  3.724734  1546.461162
4      COVID_2020    har1_var  rv1_var   144  3.942688  1462.861860
2    

For the optimized ensemble:
* For 1-day horizon: $w_1$ = 1 implying that the best 1-day ensemble is just pure GJR-GARCH
* For 5-day horizon: $w_1$ = 0 implying that the best 5-day ensemble is just pure HAR-RV

i.e., the "ensemble" is actually just a model selection by horizon. Short horizon volatility is best captured by a model that reacts to recent shocks and asymmetry (GJR-GARCH) and longer horizon volatility is best captured by a model with multi-scale persistence/long memory (HAR-RV).

Optimizing convex ensemble weights between GJR-GARCH and HAR-RV using development-window QLIKE produced a degenerate solution: the 1-day horizon assigned full weight to GJR-GARCH, while the 5-day horizon assigned full weight to HAR-RV. This suggests that the two models are optimal at different forecast horizons rather than complementary within the same horizon.

# Saving the pandas dataFrame file so that I can import in other notebooks for further analysis

In [37]:
# parquet is a file format designed for tabular data. It is efficient and preserves pandas data types well

# df_all.to_parquet("volatility_forecasts.parquet", index=False)
# df = pd.read_parquet("volatility_forecasts.parquet")

df_all.to_csv("volatility_forecasts.csv", index=False)

In [38]:
df_all.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,garch5_var,gjr1_var,gjr5_var,rv_d,rv_w,rv_m,har1_var,har5_var,ens1_var,ens5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389448,4723500,-1.733113,3.003682,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641846,5741700,0.342420,0.117251,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861015,8405550,-2.388486,5.704863,...,NaN,NaN,NaN,5.704863,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545334,7503700,-1.203735,1.448977,...,NaN,NaN,NaN,1.448977,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998020,7271850,0.624753,0.390316,...,NaN,NaN,NaN,0.390316,NaN,NaN,NaN,NaN,NaN,NaN
